# TTFL Score Prediction — Exploration

In [ ]:
from models.database import SessionLocal
from ml.data.features import build_feature_matrix, build_today_features
from ml.models.predictor import TTFLPredictor

db = SessionLocal()
df = build_feature_matrix(db)
db.close()

print(f"{len(df)} rows, {df['player_id'].nunique()} players")
df.head()

In [ ]:
predictor = TTFLPredictor.load()

print("Feature importances:")
for feat, score in predictor.feature_importance().items():
    print(f"  {feat:<25} {score}")

In [ ]:
import matplotlib.pyplot as plt

scores = df["ttfl_score"].dropna()
min = scores.min()
max = scores.max()
print(f"TTFL Score range: {min:.2f} to {max:.2f}")
bins = range(int(scores.min()), int(scores.max())+2)

plt.figure(figsize=(10, 3))
plt.hist(scores, bins=bins, color="blue")

plt.title("Distribution of TTFL Scores")
plt.xlabel("TTFL Score")
plt.ylabel("Frequency")

plt.show()


## Feature Correlation Analysis

In [ ]:
import pandas as pd
from scipy import stats

FEATURES_TO_CHECK = [
    "opp_def_rating",
    "opp_pace",
    "opp_ppg",
    "opp_rpg",
    "opp_apg",
    "is_home",
    "is_back_to_back",
    "ttfl_trend_5",
    "ttfl_trend_10",
    "std_ttfl_last_10",
    "avg_minutes_last_5",
    "games_last_10d",
]
TARGET = "ttfl_delta"

rows = []
for feat in FEATURES_TO_CHECK:
    sub = df[[feat, TARGET]].dropna()
    r, p = stats.pearsonr(sub[feat], sub[TARGET])
    rows.append({"feature": feat, "r": round(r, 4), "r²": round(r**2, 4), "p_value": round(p, 4), "significant": p < 0.05})

corr_df = pd.DataFrame(rows).sort_values("r²", ascending=False).reset_index(drop=True)
corr_df

## Tonight's Predictions

In [ ]:
from datetime import date

tonight = date(2026, 3, 7)

feat_df = build_today_features(db, tonight)
print(f"Players with enough history to predict: {len(feat_df)}")
feat_df.head()

In [ ]:
predictor = TTFLPredictor.load()

feat_df["predicted_delta"] = predictor.predict(feat_df)
feat_df["predicted_ttfl"] = feat_df["avg_ttfl_season"] + feat_df["predicted_delta"]

results = (
    feat_df[["name", "team", "opponent", "is_home", "avg_ttfl_season", "ttfl_trend_5", "predicted_delta", "predicted_ttfl"]]
    .sort_values("predicted_ttfl", ascending=False)
    .reset_index(drop=True)
)
results.index += 1
for col in ["avg_ttfl_season", "ttfl_trend_5", "predicted_delta", "predicted_ttfl"]:
    results[col] = results[col].round(1)
results["matchup"] = results.apply(lambda r: f"{'vs' if r['is_home'] else '@'} {r['opponent']}", axis=1)

results[["name", "team", "matchup", "avg_ttfl_season", "ttfl_trend_5", "predicted_delta", "predicted_ttfl"]].head(20)